In [195]:
#!/usr/bin/env python
# coding: utf-8

########################################ALL IMPORTS ############################################
import pandas as pd
import numpy as np
import random
import itertools
import json
import pprint
import datetime
from datetime import date, time, timedelta, datetime
import requests
import os
import pandas_market_calendars as mcal
import base64
from config import appKey, appSecret
from datetime import datetime, timedelta, date
import traceback
import glob

# DIFFERENT THINGS TO TEST #############
# Now we tweak to incorporate different signals to improve odds.
    
# 2) INCORPORATE THE DIFFERENT SIGNALS AND TRY THE THINGS BELOW:
    
# I could probably try all of these in another for loop of 5 binary variables and then encode them into the signals portion of the code. Just like how  i tested all the strategies, i will need to write code to test all the different signals.

# 1) 10m bars, 15m bars, 1 hr?
# 2) Spike must be higher than both any other open, or any other low, or any other high?
# 3) INCORPORATE VOLATILITY, implied vol discount, momentum, other market factors
# 4) Create market cap, float buckets, pre-market volume, and bar volume to see patterns emerge and perform backwards      looking analysis
# 5) TRY DIFFERENT TARGET ENTRIES (VWAP, VWAP_STD, VWAP*95%
# 6) PREMARKET HIGH no buy?
# 7) FULL GREEN BAR (MEASURE TO CLOSE MUST BE HIGHER THAN ALL VALUES OF DAY)(PROB NOT)
# 8) THE HIGH OF VOLUME SPIKE MUST BE HIGHER THAN ALL THE OTHER CLOSES OF THE DAY
# 9) DIFFERENT EXITS (stop and target to be set based on volatility of stock? Or on rolling volatility?
# 10) TIMING THRESHOLDS: 1) vol spike signal before x 2) buy before x 3) sell before x
# 11) BACKTEST ENHANCEMENTS FOR FUTURE STRATEGIES

# 12)DON'T BUY IF ACCOUNT_SIZE IS TOO SMALL TO BUY
# 13)CREATE IN-HOUSE FUNCTIONS


In [196]:
## DEFINING ALL VARIABLES AND THINGS POSSIBLE TO TOGGLE FOR BACKTESTING

######################################## STRATEGY NOTE TO REMIND ON OUTPUT ########################################
strategy_note = 'test_10_31'
run = strategy_note

########################################  IMPORT LIST OF ALL TICKERS FOR BACKTEST/STRATEGY   #################################
# ticker_list = pd.read_excel('./prod_files/by_float/group_2_float_11M_to_26M.xlsx')
ticker_list = pd.concat(map(pd.read_excel, glob.glob("./prod_files/by_float/*.xlsx")))
ticker_list = ticker_list[ticker_list['Float'] > 50000000]

####################################### SETTING CANDLE TIME FRAME FOR STRATEGY ########################################

# Define variables for stock time frame for strategy
period_type = 'day'  # Current value
period = 10          # Current value
frequency_type = 'minute'  # Current value
frequency = 30      # Current value
need_extended_hours_data = 'true'  # Current value
need_previous_close = 'true'  # Current value

######################################## TESTING COMBINATION OF INPUTS FOR STRATEGY  ########################################

TOTAL_CASH = 100000 #FIXED
BET_SIZE = [.1] #FIXED
STOP = [.1,.15,1] #FIXED #Removed .05
TARGET = [.05,.1] #FIXED

VOL_SPIKE_THRESHOLD = [5,10] #Abnormally high volume that stands out on a chart
PRICE_SPIKE_THRESHOLD = [.05,.1] #Must move the price x%
TIME_SIG_THRESHOLD = [time(hour=12,minute=30,second=0)]
BUY_TIME_THRESHOLD = [ time(hour=10,minute=30,second=0)]
SELL_TIME_THRESHOLD = [time(hour = 15,minute = 30,second =0)]

####################################### CREATE VARIABLES FOR INPUT STRATEGY TO TEST ##########################

bet_size_index = 0
stop_index = 1
target_index = 2
vol_spike_thresh_index = 3
price_spike_thresh_index = 4
time_sig_thresh_index = 5
buy_time_threshold_index = 6
sell_time_threshold_index = 7

variables = [BET_SIZE,STOP,TARGET,VOL_SPIKE_THRESHOLD,PRICE_SPIKE_THRESHOLD,TIME_SIG_THRESHOLD,BUY_TIME_THRESHOLD, SELL_TIME_THRESHOLD]
combinations = list(itertools.product(*variables))

# combinations = [[.1,.15,1,5,.05,time(hour = 11,minute = 30, second = 0),time(hour = 9,minute = 30, second = 0),time(hour = 9,minute = 30, second = 0)],
#                  [.1,.15,1,5,.05,time(hour = 11,minute = 30, second = 0),time(hour = 9,minute = 30, second = 0),time(hour = 10,minute = 30, second = 0)]]
#                  [.1,1,1,15,.05,time(hour = 12,minute = 30, second = 0),time(hour = 10,minute = 30, second = 0),time(hour = 10,minute = 30, second = 0)]]
#                

######################################## STRATEGY OUTPUT TABLE ########################################

#FINAL OUTPUT TO EVALUATE DIFFERENT STRATEGY COMBINATIONS

inputs = pd.DataFrame(columns=['Account Size',
                               'Bet_Size',
                               'Stop',
                               'Target',
                               'Vol Spike Thresh',
                               'Price Spike Thresh',
                               'Signal Time Threshold',
                               'Buy Time Threshold',
                               'Sell_Time',
                               'Signals',
                               'Buys',
                               'Win %',
                               'Average Win',
                               'Strategy Note'])

########################################  TIME FRAME  ########################################

# Set date parameters and calculations regarding dates such as 10 day rolling average volume

en = datetime.now()
st = en - timedelta(days=15)

days = (en-st).days

start_time = str(int(st.timestamp())*1000)
end_time = str(int(en.timestamp())*1000)

print(start_time,end_time,test_time)

today = date.today()

time_interval = 30
time_interval_string = str(time_interval)+'m'
rolling_window_days = 10
tickers_per_day = 60/time_interval*6.5
rolling_lookback = rolling_window_days*tickers_per_day

# Find Next Business Day
# Import the New York Stock Exchange Calendar
nyse = mcal.get_calendar('NYSE')
open_close_schedule = pd.DataFrame(nyse.schedule(start_date=st, end_date=en))
open_close_schedule.index.names = ['Date']
open_close_schedule.reset_index(inplace=True)
open_close_schedule['Date'] = open_close_schedule['Date'].dt.date
open_close_schedule['market_open'] = open_close_schedule['market_open'] - timedelta(hours=4)
open_close_schedule['market_close'] = open_close_schedule['market_close'] - timedelta(hours=4, minutes=time_interval)
open_close_schedule['market_open'] = open_close_schedule['market_open'].dt.time
open_close_schedule['market_close'] = open_close_schedule['market_close'].dt.time

nyse_days = nyse.valid_days(start_date=st, end_date=en)
valid_trading_days = pd.Series(nyse_days).dt.date

schedule = pd.DataFrame(nyse.schedule(start_date=st, end_date=en))


1729128842000 1730424842000 1681528372000


In [197]:
############################################## ALL FUNCTIONS ########################################################

def next_business_day(today):
    next_day = today + timedelta(days=1)
    while next_day.weekday() in [5,6] or next_day.weekday() not in valid_trading_days:
        next_day += timedelta(days=1)
    return next_day

######################################## GET BUY/SELL SIGNAL FUNCTION ##################################################


#MUST MAKE SURE THAT THE SIGNALS RETURNED MATCH UP WITH THE BUY AND SELL FUNCTION COMMANDS
#Generates the type of buy or sell signals used in the buying and selling functions by taking in a row of stock data
def buy_sell_signal(signal,
                    ok_to_buy,
                    time,
                    high_price,
                    low_price,
                    after_hours,
                    day_close,
                    date,
                    stop,
                    target,
                    sell_time_thresh,
                    buy_time_thresh,
                    yesterday_high,
                    premarket_high,
                    target_entry_price):
    
    #SHORT STRATEGIES
    if(signal == 'Short'): 
        
        #BUY (check that all signals triggered, positioning is neutral, not after hours and we didn't buy it today yet)
        if ((ok_to_buy == 'True') 
            & (time <= buy_time_thresh) 
            & (buy_time_thresh <= sell_time_thresh)
            & (high_price>=target_entry_price) 
            & (POSITION == 'Neutral') 
            & (after_hours==False) 
            & (BOUGHT_TODAY != date) 
            & (yesterday_high >= premarket_high)):
            #IF 
            #print('short',date)
            return 'Short (Target Price Cross)'
        #SELL (check that the position)
        elif (ENTRY_PRICE+(ENTRY_PRICE*stop) <= high_price) & ((POSITION == 'Short') | (POSITION == 'Long')): 
            #print('sell',date)
            return 'Sell (Stop Loss)'
        elif (ENTRY_PRICE-(ENTRY_PRICE*target) >= low_price) & ((POSITION == 'Short') | (POSITION == 'Long')):
            #print('sell',date)
            return 'Sell (Target Hit)'
        elif (((sell_time_thresh == time) | (day_close == True)) & (POSITION == 'Short') | (POSITION == 'Long')): 
            #print('sell',date)
            return 'Sell (At Time Threshold)'
        else:
            return 'none'
    if(signal == 'Long'):
        return 'none' #input long strategy signal conditions
    else:
        return 'none'
    
####################################### CALCULATE TRADE STATS ########################################################

#Returns details for results table based on the type of sell signal (profit,exit,win/loss,type of sale)
#might need entry_price passed in
def calculate_sell_results(sell_signal,row_close,position_size,stop,target):
    
    if (signal == 'Sell (Stop Loss)'):
        profit = (position_size - (position_size*(1+stop)))
        exit = ENTRY_PRICE +(ENTRY_PRICE*stop)
        win_loss = 'Loss'
        sell_type = 'Sell (Stop Loss)'
        return [profit,exit,win_loss,sell_type]

    if (signal == 'Sell (Target Hit)'):
        profit = ((1+target)*(position_size))-(position_size)
        exit = ENTRY_PRICE - (ENTRY_PRICE*target)
        win_loss = 'Win'
        sell_type = 'Sell (Target Hit)'
        return [profit,exit,win_loss,sell_type]

    if (signal == 'Sell (At Time Threshold)'):
        profit = (((ENTRY_PRICE-row_close)/ENTRY_PRICE)*position_size)
        exit = row_close
        win_loss = np.where(((ENTRY_PRICE-row['Close'])/ENTRY_PRICE)>0,'Win','Loss')
        sell_type = 'Sell (At Time Threshold)'
        return [profit,exit,win_loss,sell_type]

    
######################################### BUY AND SELL FUNCTIONS ########################################################
#Function that ultimately buys and sells stock based on signals output by the buy_sell_signal function
def buy_sell(signal,
             date,
             ticker,
             open_price,
             close_price,
             time,
             volume,
             previous_day_close,
             volume_spike,
             price_spike,
             target_entry_price,
             position_size,
             stop,
             target):
   
    global ENTRY_PRICE
    global POSITION
    global ENTRY_TIME
    global ACCOUNT_SIZE
    global BUYS
    global RESULT_INDEXER
    global BOUGHT_TODAY

    #BUY - Currently the only buy signal, but could create function for the different buy types
    if (signal == 'Short (Target Price Cross)'):

        POSITION = 'Short'
        ENTRY_TIME = time
        ENTRY_PRICE = np.where(open_price>target_entry_price,open_price,target_entry_price)
        BUYS += 1

    #SELL - Will calculate sell stats based on the signal using the calculate_sell_results function 
    elif ((signal == 'Sell (Target Hit)')
         |(signal == 'Sell (Stop Loss)')
         |(signal == 'Sell (At Time Threshold)')):
        POSITION = 'Neutral'
        sell_outputs = calculate_sell_results(signal,close_price,position_size,stop,target)
        profit = sell_outputs[0]
        exit = sell_outputs[1]
        win_loss = sell_outputs[2]
        sell_type = sell_outputs[3]
        results.at[RESULT_INDEXER,'Profit'] = profit
        results.at[RESULT_INDEXER,'Profit %'] = profit/position_size
        results.at[RESULT_INDEXER,'Bet Size'] = position_size
        ACCOUNT_SIZE += profit
        results.at[RESULT_INDEXER,'Date'] = date
        results.at[RESULT_INDEXER,'Ticker'] = ticker
        results.at[RESULT_INDEXER,'Account Size'] = ACCOUNT_SIZE
        results.at[RESULT_INDEXER,'Win/Loss'] = win_loss
        results.at[RESULT_INDEXER,'Target Entry'] = float(target_entry_price)
        results.at[RESULT_INDEXER,'Entry'] = ENTRY_PRICE  
        results.at[RESULT_INDEXER,'Exit'] = exit    
        results.at[RESULT_INDEXER,'Entry Time'] = ENTRY_TIME
        results.at[RESULT_INDEXER,'Exit Time'] = time
        results.at[RESULT_INDEXER,'Volume'] = volume
        results.at[RESULT_INDEXER,'Type'] = sell_type
        results.at[RESULT_INDEXER,'Volume Spike'] = volume_spike
        results.at[RESULT_INDEXER,'Price Spike'] = price_spike
        results.at[RESULT_INDEXER,'Previous Day Close'] = previous_day_close
        #results.at[RESULT_INDEXER,'Premarket Volume'] = 
        #results.at[RESULT_INDEXER,'Premarket Change'] = 
        BOUGHT_TODAY = date
        RESULT_INDEXER += 1

######################################### SCHWAB API FUNCTIONS ########################################################
def get_schwab_access_token():
    authUrl = f'https://api.schwabapi.com/v1/oauth/authorize?client_id={appKey}&redirect_uri=https://127.0.0.1'
    print(f"Click to authenticate: {authUrl}")
    returnedLink = input("Paste the redirect URL here:")
    code = f"{returnedLink[returnedLink.index('code=')+5:returnedLink.index('%40')]}@"

    headers = {
        'Authorization': f'Basic {base64.b64encode(bytes(f"{appKey}:{appSecret}", "utf-8")).decode("utf-8")}',
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    data = {
        'grant_type': 'authorization_code',
        'code': code,
        'redirect_uri': 'https://127.0.0.1'
    }

    response = requests.post('https://api.schwabapi.com/v1/oauth/token', headers=headers, data=data)
    token_data = response.json()
    return token_data['access_token']


def auto_authenticate():
    token_file = 'schwab_token.json'
    
    # Check if token file exists and is not expired
    if os.path.exists(token_file):
        with open(token_file, 'r') as f:
            token_data = json.load(f)
        
        expires_at = datetime.fromisoformat(token_data['expires_at'])
        if expires_at > datetime.now():
            return token_data['access_token']
    
    # If no valid token, authenticate
    headers = {
        'Authorization': f'Basic {base64.b64encode(bytes(f"{appKey}:{appSecret}", "utf-8")).decode("utf-8")}',
        'Content-Type': 'application/x-www-form-urlencoded'
    }
    data = {
        'grant_type': 'client_credentials',
        'scope': 'openid'
    }

    response = requests.post('https://api.schwabapi.com/v1/oauth/token', headers=headers, data=data)
    if response.status_code != 200:
        raise Exception(f"Authentication failed: {response.text}")

    token_data = response.json()
    # Convert expires_in to an integer before using it
    expires_in = int(token_data.get('expires_in', 3600))  # Default to 1 hour if not present
    token_data['expires_at'] = (datetime.now() + timedelta(seconds=expires_in)).isoformat()

    # Save token data
    with open(token_file, 'w') as f:
        json.dump(token_data, f)

    return token_data['access_token']

  

def get_stock_price_history(symbol, access_token, period_type, period, frequency_type, frequency, start_date, end_date, need_extended_hours_data, need_previous_close):
    url = 'https://api.schwabapi.com/marketdata/v1/pricehistory'
    params = {
        'symbol': symbol,
        'periodType': period_type,
        'period': period,
        'frequencyType': frequency_type,
        'frequency': frequency,
        'startDate': start_date,
        'endDate': end_date,
        'needExtendedHoursData': need_extended_hours_data,
        'needPreviousClose': need_previous_close
    }
    headers = {'Authorization': f'Bearer {access_token}'}
    
    response = requests.get(url, params=params, headers=headers)
    return response.json()

    #DOCUMENTATION FOR API CALL
    # If the periodType is
    # • day - valid values are 1, 2, 3, 4, 5, 10
    # • month - valid values are 1, 2, 3, 6
    # • year - valid values are 1, 2, 3, 5, 10, 15, 20
    # • ytd - valid values are 1

    # If the period is not specified and the periodType is
    # • day - default period is 10.
    # • month - default period is 1.
    # • year - default period is 1.
    # • ytd - default period is 1.

    # period
    # frequencyType
    # string
    # (query)
    # The time frequencyType

    # If the periodType is
    # • day - valid value is minute
    # • month - valid values are daily, weekly
    # • year - valid values are daily, weekly, monthly
    # • ytd - valid values are daily, weekly

    # If frequencyType is not specified, default value depends on the periodType
    # • day - defaulted to minute.
    # • month - defaulted to weekly.
    # • year - defaulted to monthly.
    # • ytd - defaulted to weekly.

    # Available values : minute, daily, weekly, monthly


    # --
    # frequency
    # integer($int32)
    # (query)
    # The time frequency duration

    # If the frequencyType is
    # • minute - valid values are 1, 5, 10, 15, 30
    # • daily - valid value is 1
    # • weekly - valid value is 1
    # • monthly - valid value is 1

    # If frequency is not specified, default value is 1

    # frequency
    # startDate
    # integer($int64)
    # (query)
    # The start date, Time in milliseconds since the UNIX epoch eg 1451624400000
    # If not specified startDate will be (endDate - period) excluding weekends and holidays.

    # startDate
    # endDate
    # integer($int64)
    # (query)
    # The end date, Time in milliseconds since the UNIX epoch eg 1451624400000
    # If not specified, the endDate will default to the market close of previous business day.

    # Can use this API Wrapper or take as inspiration: https://github.com/tylerebowers/Schwab-API-Python/blob/main/docs/stream.md
####Create Function for input stats to track to easily just change the list
# # all STRATEGY INPUTS AND VARIABLES should be put into different config files



In [198]:
# Download data into memory
# Ensure the 'Tickers' column exists
if 'Ticker' in ticker_list.columns:
    # Create a new DataFrame with only the 'Tickers' column
    tickers_df = ticker_list[['Ticker']].dropna()
    # Convert to a list of unique tickers
    tickers = tickers_df['Ticker'].unique().tolist()  # Extract unique tickers
    print(tickers)  # Display the list of tickers
    print('SUCESS IF YOU SEE TICKERS!!! ^^^^^^^')
else:
    print("Column 'Ticker' not found in the DataFrame.")

['BL', 'ALKT', 'STEW', 'CRL', 'ZBRA', 'PASG', 'AX', 'KFY', 'JKS', 'HCC', 'MOD', 'LMND', 'AIZ', 'TRMD', 'SNA', 'THS', 'AREC', 'CENX', 'VSTO', 'GDOT', 'WFG', 'DRTS', 'WDI', 'TY', 'SGML', 'CBU', 'CLYM', 'EBS', 'MHK', 'HG', 'VBTX', 'PLTK', 'MGTX', 'TFSL', 'GRNT', 'OPI', 'RGEN', 'VTEX', 'ACP', 'MHD', 'ZG', 'PII', 'ZVRA', 'EOS', 'REPL', 'KLIC', 'ARW', 'HYPR', 'MLTX', 'SOL', 'CENTA', 'QNST', 'FULC', 'PCN', 'SANM', 'PRQR', 'SUPN', 'IQI', 'RGLS', 'GKOS', 'DXLG', 'IDA', 'NNOX', 'SLRC', 'BTSG', 'MIDD', 'HLI', 'MATV', 'PRIM', 'PAVS', 'EXFY', 'CRUS', 'MUJ', 'CION', 'VUZI', 'CNTB', 'BAP', 'FIVE', 'WHR', 'WIX', 'EPAC', 'HUBB', 'BHAT', 'CRTO', 'ARBK', 'PLRX', 'CHW', 'PRTS', 'PHR', 'CBT', 'AHT', 'BHK', 'EMBC', 'ELF', 'LIVN', 'NDSN', 'LECO', 'REYN', 'VGM', 'GRWG', 'PAM', 'BUSE', 'FUL', 'ACB', 'EOLS', 'ENV', 'RYTM', 'CTOS', 'KTB', 'BZUN', 'MMU', 'IVR', 'RS', 'IBOC', 'VTYX', 'TDG', 'PXLW', 'GPCR', 'PARR', 'NMCO', 'RNST', 'NMRA', 'ONL', 'LUNR', 'PTLO', 'EPAM', 'ABVX', 'ENSG', 'AGIO', 'ENOV', 'VNDA', 'RA', 

In [199]:
# ########################################  SET SINGLE TEST TICKER   #################################
# # For initial testing, we'll just use a single ticker
# tickers = ['AAPL']  # Can easily change this to test different tickers
# print(f"Testing strategy on ticker: {tickers[0]}")

In [200]:
################################## TESTING/AUTHENTICATING CONNECTION TO SCHWAB API ##################################
# Get access token (figure out how often to do this optimally)
access_token = auto_authenticate()

# Get AAPL price history for the last year using the defined variables
aapl_price_history = get_stock_price_history('AAPL', access_token, period_type, period, frequency_type, frequency,  start_time, end_time,need_extended_hours_data, need_previous_close)
print(start_time,end_time)

# Convert to DataFrame
aapl_df = pd.DataFrame(aapl_price_history['candles'])

# Rename columns to match the required format
aapl_df.rename(columns={
    'datetime': 'Datetime',
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
}, inplace=True)

# Convert Datetime to EST
aapl_df['Datetime'] = pd.to_datetime(aapl_df['Datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)

# Save the DataFrame as a CSV file
aapl_df.to_csv('./file_uploads/tests/aapl_price_history.csv', index=False)

# Print the result

print(aapl_price_history)
print('SUCCESS IF I SEE APPLE STOCK DATA!!!')
print('AAPL price history saved to ./file_uploads/tests/aapl_price_history.csv')

1729128842000 1730424842000
{'candles': [{'open': 233.69, 'high': 234.23, 'low': 233.69, 'close': 234.19, 'volume': 9625, 'datetime': 1729076400000}, {'open': 234.2, 'high': 234.22, 'low': 233.85, 'close': 233.958, 'volume': 11820, 'datetime': 1729078200000}, {'open': 234.04, 'high': 234.04, 'low': 232.84, 'close': 233.0, 'volume': 79014, 'datetime': 1729080000000}, {'open': 232.9513, 'high': 233.02, 'low': 232.84, 'close': 232.91, 'volume': 46418, 'datetime': 1729081800000}, {'open': 232.91, 'high': 233.23, 'low': 231.4, 'close': 231.7, 'volume': 242058, 'datetime': 1729083600000}, {'open': 231.71, 'high': 232.12, 'low': 230.03, 'close': 230.08, 'volume': 5197930, 'datetime': 1729085400000}, {'open': 230.085, 'high': 230.99, 'low': 229.84, 'close': 230.7297, 'volume': 2487237, 'datetime': 1729087200000}, {'open': 230.71, 'high': 230.77, 'low': 230.05, 'close': 230.3995, 'volume': 1542566, 'datetime': 1729089000000}, {'open': 230.4, 'high': 231.145, 'low': 230.3401, 'close': 230.585, '

In [201]:

######################### CREATE INITIAL DATAFRAME OF STOCK PRICE HISTORY AND PERFORM PRECALCULATIONS #########################

start_clock = datetime.now()  # calculate run time
go = 1
stockies = {} #Create dataframes of stock data for iteration

for ticker in tickers:
    # Get stock price history from Schwab API
    stock_data = get_stock_price_history(ticker, access_token, period_type, period, frequency_type, frequency, start_time, end_time, need_extended_hours_data, need_previous_close)
    
    if not stock_data or 'candles' not in stock_data:
        print(f"No data available for {ticker}")
        continue
    
    # Convert to DataFrame
    stahks = pd.DataFrame(stock_data['candles'])
    
    # Ensure all required fields are present
    required_fields = ['datetime', 'open', 'high', 'low', 'close', 'volume']
    if not all(field in stahks.columns for field in required_fields):
        print(f"Missing required fields for {ticker}. Available columns: {stahks.columns}")
        continue
    
    # Rename columns to match the required format
    stahks.rename(columns={
        'datetime': 'Datetime',
        'open': 'Open',
        'high': 'High',
        'low': 'Low',
        'close': 'Close',
        'volume': 'Volume'
    }, inplace=True)
    
    # Convert Datetime to EST
    stahks['Datetime'] = pd.to_datetime(stahks['Datetime'], unit='ms').dt.tz_localize('UTC').dt.tz_convert('US/Eastern').dt.tz_localize(None)
    
    # Print the DataFrame to inspect its structure
    # print(f"Data for {ticker}:")
    # print(stahks.head())  # Display the first few rows of the DataFrame
    # print("Columns in DataFrame:", stahks.columns)  # Print the column names
    
    #Skip stock if there is insufficient amount of data (data for each period of normal trading hours)
    end_check = stahks['Datetime'].max()
    start_check = stahks['Datetime'].min()
    daydiff = end_check.weekday() - start_check.weekday()
    days = ((end_check-start_check).days - daydiff) / 7 * 5 + min(daydiff,5) - (max(end_check.weekday() - 4, 0) % 5)
    
    print(ticker, len(stahks.index))    
    # Removed the erroneous line that referenced 'datetime' instead of 'Datetime'
    # stahks['datetime'] = pd.to_datetime(stahks['datetime']/1000, unit = 's')-timedelta(hours =4)
    # stahks.columns = ['Open','High','Low','Close','Volume','Datetime']
    
    #Rearrange Columns and Merge with Open/Close Schedule
    stahks['Ticker'] = ticker
    stahks['Date'] = stahks['Datetime'].dt.date
    stahks = stahks.merge(open_close_schedule,how = 'left', on = 'Date')

    # Only calculate and average volume if there is sufficient data
    rolling_lookback_int = int(rolling_lookback)
    stahks['10_Day_Avg_Vol'] = stahks.Volume.rolling(rolling_lookback_int, min_periods=rolling_lookback_int).mean()
    stahks['10_Day_Avg_Vol'] = stahks['10_Day_Avg_Vol'].fillna(float('inf'))
    stahks['Time'] = stahks['Datetime'].dt.time
    
    #Remove any accidental duplicates (FIGURE OUT WHY????)
    stahks.drop_duplicates(['Ticker','Date','Time'],inplace = True,ignore_index=True)
    
    if len(stahks.index) < (days * tickers_per_day):
        continue
        
    #Create column with the open bar's low price (For % gap up calculation with spike later in day)
    cond = (stahks['Time'] == stahks['market_open'])
    stahks['Day_Open_Low'] = stahks[cond].groupby('Date',as_index=True)['Low'].transform('min').ffill()
    stahks['After Hours'] = (stahks['Time'] > stahks['market_close']) | (stahks['Time'] < stahks['market_open'])
    cond_2 = (stahks['After Hours'] == True)
    stahks['Pre-Market High'] = stahks[cond_2].groupby('Date',as_index=True)['High'].transform('max')
    
    stahks = stahks.ffill(axis=0)
    stahks = stahks.bfill(axis=0)

    
    stahks['VWAP_Row'] = stahks['Volume']*((stahks['High']+stahks['Low']+stahks['Close'])/3)
    stahks['Cum_VWAP'] = stahks.groupby('Date')['VWAP_Row'].transform('cumsum')
    stahks['Cum_Volume'] = stahks.groupby('Date')['Volume'].transform('cumsum')
    stahks['VWAP'] = stahks['Cum_VWAP']/stahks['Cum_Volume']
    stahks['VWAP_STD_1'] = stahks['VWAP'] - stahks.groupby('Date')['VWAP'].transform('std')
    stahks['Color_Bar'] = np.where(stahks['Open']<=stahks['Close'], 'Green', 'Red')
    #vol_window = 1
    stahks['Day_Close'] = (stahks['Time'] == stahks['market_close'])

    
    stockies[ticker] = pd.DataFrame(stahks, columns=stahks.keys())
    print(f"{ticker} processed successfully.")
    go += 1

    #Calculate pre-market volume for day 
    #Calculate pre-market change for the day 
    #stockies['Ticker_Return'] = (stahks['Close']/stahks['Close'].shift(vol_window))-1
    #stockies['Rolling_Vol'] = stahks['Ticker_Return'].std(ddof=130)
    
    ##PRINT OUT THE DATAFRAME TO A CSV FILE
    # stahks.to_csv(f"results/test/{ticker}_stahks_data.csv", index=False, header=True)

#Stockies is the final dataframe that will be used for the backtest




BL 175
BL processed successfully.
ALKT 199
ALKT processed successfully.
STEW 170
STEW processed successfully.
CRL 194
CRL processed successfully.
ZBRA 178
ZBRA processed successfully.
PASG 200
PASG processed successfully.
AX 181
AX processed successfully.
KFY 175
KFY processed successfully.
JKS 343
JKS processed successfully.
HCC 192
HCC processed successfully.
MOD 192
MOD processed successfully.
LMND 259
LMND processed successfully.
AIZ 182
AIZ processed successfully.
TRMD 301
TRMD processed successfully.
SNA 183
SNA processed successfully.
THS 172
THS processed successfully.
AREC 199
AREC processed successfully.
CENX 196
CENX processed successfully.
VSTO 182
VSTO processed successfully.
GDOT 179
GDOT processed successfully.
WFG 171
WFG processed successfully.
DRTS 87
WDI 179
WDI processed successfully.
TY 156
TY processed successfully.
SGML 204
SGML processed successfully.
CBU 172
CBU processed successfully.
CLYM 174
CLYM processed successfully.
EBS 250
EBS processed successfully.
MH

In [193]:
######################### START OF BACKTESTING ########################################################

input_indexer = 1

for strategy in combinations:
    #Skips combos where the buy time threshold is after the sell time threshold
    if (strategy[buy_time_threshold_index]>strategy[sell_time_threshold_index]):
        continue           
    
    print(input_indexer, datetime.now())
    
########################################### DEFINING BACKTEST RESULTS TABLE FOR EACH VARIATION OF STRATEGY ########################################

    results = pd.DataFrame(columns=['Ticker',
                                    'Date',
                                    'Volume Spike',
                                    'Price Spike',
                                    'Previous Day Close',
                                    'Signal Time',
                                    'Target Entry',
                                    'Entry Time',
                                    'Exit Time',
                                    'Account Size',
                                    'Bet Size',
                                    'Win/Loss', 
                                    'Profit',
                                    'Profit %',
                                    'Entry',
                                    'Exit',
                                    'Type',
                                    'Volume',
                                    'Premarket Volume',
                                    'Premarket Change'])
                                    
    RESULT_INDEXER= 0
    #Create temporary variable to output total number of signals and purchases found (where we'd need to be prepared to buy on the next day)
    SIGNALS = 0
    BUYS = 0
    ACCOUNT_SIZE = TOTAL_CASH

    ################################################   IMPORT DATA   ##################################################         
    
    tickers = list(stockies.keys())
    random.shuffle(tickers)
    
    for ticker in tickers:

    ###########################################  CALCULATE SIGNALS FOR BACKTEST BASED ON STRATEGY INPUTS  ###########################################

        #Checks for Highest Daily Volume
        high_vol_sig = np.where(stockies[ticker]['Volume'] == stockies[ticker].groupby('Date')['Volume'].transform('max'),'True','False')
        stockies[ticker]['high_vol_sig'] = high_vol_sig
        #Checks for price spike more than X% greater than 
        price_sig = np.where((stockies[ticker]['High']-stockies[ticker]['Day_Open_Low'])/stockies[ticker]['Day_Open_Low'] >= strategy[price_spike_thresh_index],'True','False')
        stockies[ticker]['price_sig'] = price_sig
        #Checks that the spike was before X time threshold (tied to highest daily volume)
        before_time_thresh = np.where(stockies[ticker]['Time'] <= strategy[time_sig_thresh_index], 'True','False')
        stockies[ticker]['before_time_thresh'] = before_time_thresh
        #Checks that spike was a green bar
        green_bar = np.where(stockies[ticker]['Color_Bar'] == 'Green', 'True','False')
        stockies[ticker]['green_bar'] = green_bar
        #Checks for volume spike X% greater than 10day average
        vol_spike_sig = np.where(stockies[ticker]['Volume'] > strategy[vol_spike_thresh_index] * stockies[ticker]['10_Day_Avg_Vol'],'True','False')
        stockies[ticker]['vol_spike_sig'] = vol_spike_sig
        #Checks that the spike high was the highest price of the day
        high_price_sig = np.where(stockies[ticker]['High'] >= stockies[ticker].groupby('Date')['Close'].transform('max'),'True','False')
        stockies[ticker]['high_price_sig'] = high_price_sig
        #Checks to see if it time is before equal to the sell_time threshold set by user
        stockies[ticker]['sell_time'] = np.where(stockies[ticker]['Time'] == strategy[sell_time_threshold_index],'True','False')
        #Finds High of the Day
        stockies[ticker]['high_of_day'] = stockies[ticker].groupby('Date')['High'].transform('max')
        #Checks all conditions
        stockies[ticker]['Grab_Price_Signal'] = np.where((high_vol_sig == 'True') 
                                                         & (vol_spike_sig == 'True') 
                                                         & (price_sig == 'True') 
                                                         & (high_price_sig == 'True') 
                                                         #& (green_bar == 'True') 
                                                         & (before_time_thresh == 'True'),
                                                         'True','False')
        
    ################################ INPUT BUY AND SELL SIGNALS FOR BACKTEST ##################################################

        #Find a way to turn on and off signals and rules to be 'True' 'False' 'Ignore' - yet still works with backtest framework

        #Checks that the close was lower than the VWAP (maybe in backtest)
        stockies[ticker]['Close_Condition'] = 'False'
        #Checks that all criteria was checked (X days ago) and that we are just waiting for buy signal (price crosses above VWAP)
        stockies[ticker]['Ok_To_Buy'] = 'False'
        #Creates the Buy and Sell Signal Column for iteration
        stockies[ticker]['Buy_Sell_Signal'] = 'None'
        #Create target entry price column
        stockies[ticker]['Target_Entry_Price'] = 100000.0

        #Temp Variables for backtest (row by row iteration)
        #Create temporary signal day variable that signaled whether or not the price_to_buy_signal was triggered during that day
        TEMP_SIGNAL_DAY = stockies[ticker]['Date'][0] - timedelta(days=1)
        #Create temporary ok to buy day variable that will trigger if the close condition for the day was satisfied (which only triggers if grab_price_signal is triggered)
        OK_TO_BUY_DAY = stockies[ticker]['Date'][0] - timedelta(days=2)
        #Initially set not to trigger and gets set on price_buy_signal
        TARGET_ENTRY_PRICE = 0.0 
        ENTRY_PRICE = 0.0 
        ENTRY_TIME = stockies[ticker]['Time'][0]
        BOUGHT_TODAY = stockies[ticker]['Date'][0] - timedelta(days=1)
        PREVIOUS_DAY_CLOSE = 0.0
        SIGNAL_TIME = stockies[ticker]['Time'][0]
        OPTIMAL_ENTRY = 0.0
        OPTIMAL_EXIT = 0.0
        VOLUME_SPIKE = 0.0
        PRICE_SPIKE_TEMP = 0.0
        YESTERDAY_HIGH = 0.0

        OPTIMAL_ENTRY_TIME = 0.0
        OPTIMAL_EXIT_TIME = 0.0

        POSITION = 'Neutral'

        # Save stockies to see its structure
        stockies[ticker].to_excel('./results/test/stockies_structure.xlsx', index=True, header=True)

        #Iterate over rows to see which rows meet the close condition and the all clear to buy signal (pending final signal: price cross)
        #Unique to this strategy's backtest. Could be a part of inserting variables and signals before BACKTEST SECTION

        for index, row in stockies[ticker].iterrows():

            if row['Grab_Price_Signal'] == 'True':
                TEMP_SIGNAL_DAY = row['Date']
                TARGET_ENTRY_PRICE = row['VWAP']
                SIGNAL_TIME = row['Time']
                VOLUME_SPIKE = row['Volume']/row['10_Day_Avg_Vol']
                PRICE_SPIKE = (row['High']-row['Day_Open_Low'])/row['Day_Open_Low']
                YESTERDAY_HIGH = row['high_of_day']

            if (row['Close']<=TARGET_ENTRY_PRICE) & (row['Day_Close'] == True) & (row['Date'] == TEMP_SIGNAL_DAY):
                stockies[ticker].at[index,'Close_Condition'] = 'True'
                OK_TO_BUY_DAY = next_business_day(row['Date'])
                SIGNALS += 1
                PREVIOUS_DAY_CLOSE = row['Close']
            
            if (OK_TO_BUY_DAY == row['Date']):
                
                stockies[ticker].at[index,'Ok_To_Buy'] = 'True' 
                stockies[ticker].at[index,'Previous_Day_Close'] = PREVIOUS_DAY_CLOSE
                stockies[ticker].at[index,'Signal Time'] = SIGNAL_TIME
                stockies[ticker].at[index,'Volume Spike'] = VOLUME_SPIKE
                stockies[ticker].at[index,'Price_Spike_From_Open'] = PRICE_SPIKE
                stockies[ticker].at[index,'Yesterday High'] = YESTERDAY_HIGH
                stockies[ticker].at[index,'Target_Entry_Price'] = TARGET_ENTRY_PRICE
                
        #Consolidate tables to only days where we might buy and sell
        stonks = stockies[ticker][(stockies[ticker]['Ok_To_Buy'] == 'True')]
        stonks.to_excel('./results/test/stonks.xlsx', index = True, header=True)
        
        #Check that pre-market high wasn't higher than yesterday's high bar (in backtest)
        #Use variable for yesterday's high that switches on signal to equal high price of signal
        #Compare to yesterday's high bar       
        #Check that pre-market high wasn't higher than yesterday's high bar (in backtest)
        #Use variable for yesterday's high that switches on signal to equal high price of signal
        #Compare to yesterday's high bar

        #If there are no signals - skip to next stock.
        if len(stockies[ticker][stockies[ticker]['Ok_To_Buy']== 'True']) == 0:
            continue           

    #########################################  BACKTEST IMPLEMENTATION AND SIMULATION OF BUY AND SELL SIGNALS  ##################################################
        
        POSITION = 'Neutral'

        #USING IN-HOUSE FUNCTION SIMULATE (ITERATING THROUGH DAYS, BUYING AND SELLING BASED ON SIGNALS)
        #Iterate over rows to fill in results table with buy and sell actions
        for index, row in stonks.iterrows():        
            
            signal = buy_sell_signal('Short',
                                     row['Ok_To_Buy'],
                                     row['Time'],
                                     row['High'],
                                     row['Low'],
                                     row['After Hours'],
                                     row['Day_Close'],
                                     row['Date'],
                                     strategy[stop_index],
                                     strategy[target_index],
                                     strategy[sell_time_threshold_index],
                                     strategy[buy_time_threshold_index],
                                     row['Yesterday High'],
                                     row['Pre-Market High'],
                                     row['Target_Entry_Price'])
            if signal == 'none':
                continue
            position_size = ACCOUNT_SIZE * strategy[bet_size_index]    
            buy_sell(signal,
                     row['Date'],
                     row['Ticker'],
                     row['Open'],
                     row['Close'],
                     row['Time'],
                     row['Volume'],
                     row['Previous_Day_Close'],
                     row['Volume Spike'],
                     row['Price_Spike_From_Open'],
                     row['Target_Entry_Price'],
                     position_size,
                     strategy[stop_index],
                     strategy[target_index])
            
            signal = buy_sell_signal('Short',
                                     row['Ok_To_Buy'],
                                     row['Time'],
                                     row['High'],
                                     row['Low'],
                                     row['After Hours'],
                                     row['Day_Close'],
                                     row['Date'],
                                     strategy[stop_index],
                                     strategy[target_index],
                                     strategy[sell_time_threshold_index],
                                     strategy[buy_time_threshold_index],
                                     row['Yesterday High'],
                                     row['Pre-Market High'],
                                     row['Target_Entry_Price'])
            if signal == 'none':
                continue
            position_size = ACCOUNT_SIZE * strategy[bet_size_index]    
            buy_sell(signal,
                     row['Date'],
                     row['Ticker'],
                     row['Open'],
                     row['Close'],
                     row['Time'],
                     row['Volume'],
                     row['Previous_Day_Close'],
                     row['Volume Spike'],
                     row['Price_Spike_From_Open'],
                     row['Target_Entry_Price'],
                     position_size,
                     strategy[stop_index],
                     strategy[target_index])
            
        #HOW DO I CHECK FOR BUY AND SELL IN THE SAME STRATEGY (JUST COPY PASTE IT AND DO IT TWICE)
        #results['Optimal Entry'] = stockies[results['Date']==stockies['Date']].groupby('Date')['High'].transform('max')
        #results['Optimal Exit'] = stockies[results['Date']==stockies['Date']].groupby('Date')['Low'].transform('min')
        #results['Optimal Entry Time'] =stockies[results['Date']==stockies['Date']].groupby('Date')['High'].transform('max')
        #results['Optimal Exit Time'] =
        #results['Left On Table'] = ((results['Optimal Entry']*results['Bet Size'])-(results['Optimal Exit']*results['Bet Size']))-((results['Bet Size'])-(STOP*results['Bet Size']))
    
    inputs.at[input_indexer,'Account Size'] = ACCOUNT_SIZE
    inputs.at[input_indexer,'Bet_Size'] = strategy[bet_size_index]
    inputs.at[input_indexer,'Stop'] = strategy[stop_index]
    inputs.at[input_indexer,'Target'] = strategy[target_index]
    inputs.at[input_indexer,'Vol Spike Thresh'] = strategy[vol_spike_thresh_index]
    inputs.at[input_indexer,'Price Spike Thresh'] = strategy[price_spike_thresh_index]
    inputs.at[input_indexer,'Signal Time Threshold'] = strategy[time_sig_thresh_index]
    inputs.at[input_indexer,'Buy Time Threshold'] = strategy[buy_time_threshold_index]
    inputs.at[input_indexer,'Signals'] = SIGNALS
    inputs.at[input_indexer,'Buys'] = BUYS
    inputs.at[input_indexer,'Average Win'] = results['Profit %'].mean()
    inputs.at[input_indexer,'#Hit Target'] = len(results[results['Type'] == 'Sell (Target Hit)'])
    inputs.at[input_indexer,'#Hit Stop'] = len(results[results['Type'] == 'Sell (Stop Loss)'])
    inputs.at[input_indexer,'#Sold at Time']= len(results[results['Type'] == 'Sell (At Time Threshold)'])
    inputs.at[input_indexer,'Sell_Time']= strategy[sell_time_threshold_index]
    inputs.at[input_indexer,'Strategy Note']= strategy_note
    try:
        inputs.at[input_indexer,'Win %'] = (len(results[results['Win/Loss']=='Win'])/len(results['Win/Loss']))
    except:
        pass
    tick_list = tickers
    # Write each dataframe to a different worksheet.
    # results = pd.merge(results,tick_list[['Ticker','Market Capitalization','Sector','Shares Float']],on = 'Ticker', how = 'left')
    results.to_excel(f'results/detailed_results/run_{run}_guide_{input_indexer}.xlsx', index=False, header=True)
    input_indexer += 1

################################################### CLEAN OUTPUTS ####################################################

    print ("Ending Account Size: ", ACCOUNT_SIZE)
    print ("Signals: ", SIGNALS)
    print ("Buys:", BUYS)

    try:
        print ("Win %: ", len(results[results['Win/Loss']=='Win'])/len(results['Win/Loss'])*100,"%")
        print("Avg. Win: ", results['Profit %'].mean()*100,"%")

    except:
        pass

# plot = px.line(results, x = results.index.values, y = 'Account Size', title = 'Equity Curve')
# plot.show()
current_date = datetime.now().strftime("%Y-%m-%d")
inputs.to_excel(f'results/run_{run}_on_{current_date}.xlsx', index=True, header=True)
print(datetime.now() - start_clock)


1 2024-10-31 18:33:36.661747
Ending Account Size:  100000
Signals:  0
Buys: 0
2 2024-10-31 18:33:36.796213
Ending Account Size:  100000
Signals:  0
Buys: 0
3 2024-10-31 18:33:36.908747
Ending Account Size:  100000
Signals:  0
Buys: 0
4 2024-10-31 18:33:37.017127
Ending Account Size:  100000
Signals:  0
Buys: 0
5 2024-10-31 18:33:37.127488
Ending Account Size:  100000
Signals:  0
Buys: 0
6 2024-10-31 18:33:37.235941
Ending Account Size:  100000
Signals:  0
Buys: 0
7 2024-10-31 18:33:37.347348
Ending Account Size:  100000
Signals:  0
Buys: 0
8 2024-10-31 18:33:37.463996
Ending Account Size:  100000
Signals:  0
Buys: 0
9 2024-10-31 18:33:37.568782
Ending Account Size:  100000
Signals:  0
Buys: 0
10 2024-10-31 18:33:37.672670
Ending Account Size:  100000
Signals:  0
Buys: 0
11 2024-10-31 18:33:37.781910
Ending Account Size:  100000
Signals:  0
Buys: 0
12 2024-10-31 18:33:37.891327
Ending Account Size:  100000
Signals:  0
Buys: 0
13 2024-10-31 18:33:37.994128
Ending Account Size:  100000
Si

In [194]:
current_date = datetime.now().strftime("%Y-%m-%d")
inputs.to_excel(f'results/run_{run}_on_{current_date}.xlsx', index=True, header=True)